In [1]:
import numpy as np
import pandas as pd
import xsim
import xtools as xt
import matplotlib.pyplot as plt
from pyqubo import Array, Constraint
from openjij import SQASampler, SASampler
from typing import Callable, Dict, List

from atm.flight.generator import ScenarioGenerator, ScenarioIterator
from atm.optimizer.objective import get_obj_single_runway, calc_assign_time
from atm.separation import recat, faa
from atm.optimizer.problem import Problem

In [2]:
cf = xt.Config(dict(
    num=10,
    scenario=dict(
        interval=60,
        window=200,
        mode="mix",
        standard="recat",
        seed=0,
    ),
    penalty_coef=10,
))
sep = recat.TBS if cf.scenario.standard == "recat" else faa.TBS

In [3]:
prob = Problem(cf)
prob.evaluate(np.arange(cf.num))

429.8

In [40]:
class StochasticSequenceOptimizer:
    
    def __init__(self):
        self.prob = None
        self.algo = None
                
        # experiments
        # logging
        self.logger = None
        # random
        self.rs = np.random.RandomState()
        self._seed = None
    
    def seed(self, val=None):
        if val is None:
            val = np.random.randint(0, 1e+8)
        val = val + 2 ** 8
        self.rs = np.random.RandomState(val)
        self._seed = val
        return val
    
    def initialize(self):
        nx = self.prob.size
        xc = np.arange(nx)
        ec = self.prob.evaluate(xc)
        xb, eb = xc, ec

        self.logger = xsim.Logger()
        return xc, ec, xb, eb
    
    def log(self, xc, ec, xb, eb, step=None):
        if step is None:
            step = len(self.logger)
        self.logger.store(step=step, xc=xc, ec=ec, xb=xb, eb=eb).flush()
    
    def optimize(self, problem, algorithm, max_iter):
        # receive prob and algo
        algorithm.seed(self.rs.randint(0, 1e+8))
        prob = self.prob = problem
        algo = self.algo = algorithm
        
        # initialize
        nx = self.prob.size
        xc, ec, xb, eb = self.initialize()
        self.log(xc, ec, xb, eb)
        
        # main loop
        for step in range(1, max_iter + 1):
            print(f"- step: {step} - - - - - - - - - -")
            xp = algo.sample(xc, step=step)
            ep = prob.evaluate(xp)
            
            # stochastic accepting
            r = algo.calc_acceptance_prob(ep, ec)
            if r > self.rs.random():
                xc = xp
                ec = ep
                
            # update best            
            if eb >= ec:
                eb = ec
                xb = xc
            
            self.log(xc, ec, xb, eb, step=step)


In [41]:
class Algorithm:
    
    def __init__(self, temp: float = 1.):
        self.rs = np.random.RandomState()
        self._seed = None
        self.T = temp
    
    def seed(self, seed):
        if seed is None:
            seed = np.random.randint(0, 1e+8)
        self.rs = np.random.RandomState(seed)
        self._seed = seed
        return seed
    
    def sample(self, x, **kwargs):
        pass
    
    def calc_acceptance_prob(self, ec, ep, **kwargs):
        pass
    
    def exp(self, e):
        return np.exp(-1 * e / self.T)

In [42]:
class SequenceHastingsAlgorithm(Algorithm):
    
    def __init__(self, temp: float = 1.):
        super().__init__(temp=temp)
    
    def sample(self, x, **kwargs):
        idx1, idx2 = self.rs.choice(len(x), size=2, replace=False)
        xp = np.squeeze(x).copy()
        xp[idx1] = x[idx2]
        xp[idx2] = x[idx1]
        return xp
    
    def calc_acceptance_prob(self, ec, ep, **kwargs):
        ef = self.exp
        r = ef(ep) / ef(ec)
        return np.min([1., r])

In [47]:
prob = Problem(cf)
algo = SequenceHastingsAlgorithm(temp=5.)
optim = StochasticSequenceOptimizer()
optim.optimize(prob, algo, 10)

- step: 1 - - - - - - - - - -
- step: 2 - - - - - - - - - -
- step: 3 - - - - - - - - - -
- step: 4 - - - - - - - - - -
- step: 5 - - - - - - - - - -
- step: 6 - - - - - - - - - -
- step: 7 - - - - - - - - - -
- step: 8 - - - - - - - - - -
- step: 9 - - - - - - - - - -
- step: 10 - - - - - - - - - -


In [48]:
optim.logger.buffer()

{'step': array([[ 0.],
        [ 1.],
        [ 2.],
        [ 3.],
        [ 4.],
        [ 5.],
        [ 6.],
        [ 7.],
        [ 8.],
        [ 9.],
        [10.]], dtype=float32),
 'xc': array([[0., 1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [5., 1., 2., 3., 4., 0., 6., 7., 8., 9.],
        [7., 1., 2., 3., 4., 0., 6., 5., 8., 9.],
        [7., 1., 2., 3., 4., 0., 6., 5., 8., 9.],
        [7., 1., 2., 3., 4., 0., 6., 5., 8., 9.],
        [7., 1., 2., 3., 4., 9., 6., 5., 8., 0.],
        [7., 1., 2., 3., 4., 9., 6., 5., 8., 0.],
        [7., 1., 5., 3., 4., 9., 6., 2., 8., 0.],
        [7., 1., 5., 3., 4., 9., 6., 2., 8., 0.],
        [7., 2., 5., 3., 4., 9., 6., 1., 8., 0.],
        [7., 5., 2., 3., 4., 9., 6., 1., 8., 0.]], dtype=float32),
 'ec': array([[429.8],
        [513.6],
        [624.3],
        [624.3],
        [624.3],
        [625.9],
        [625.9],
        [646.2],
        [646.2],
        [643.3],
        [664.3]], dtype=float32),
 'xb': array([[0., 1., 2., 